# Se obtienen los dato relacionados a las estaciones, se tranforma a un Dataframe y se guarda

In [1]:
import pandas as pd

# URL del feed GBFS
url = "https://gbfs.lyft.com/gbfs/2.3/chi/en/station_information.json"

# Leer directamente el JSON
data = pd.read_json(url)

# La información de las estaciones está dentro de data["data"]["stations"]
stations = pd.DataFrame(data["data"]["stations"])

print(stations.dtypes)
print(stations.head())
print(stations.shape)

name            object
capacity         int64
lon            float64
lat            float64
short_name      object
station_id      object
rental_uris     object
region_id       object
address         object
dtype: object
                       name  capacity        lon        lat short_name  \
0  Kildare Ave & Archer Ave        15 -87.730790  41.800340   CHI01978   
1  California Ave & 36th St        15 -87.694684  41.828098   CHI01875   
2   Long Ave & Diversey Ave        15 -87.761240  41.931040   CHI01932   
3  Meade Ave & Diversey Ave        15 -87.778450  41.930860   CHI01840   
4     Wood St & Cortland St        11 -87.672810  41.915890   CHI02134   

            station_id                                        rental_uris  \
0  1963672425916463636  {'android': 'https://chi.lft.to/lastmile_qr_sc...   
1  1934289361049585738  {'android': 'https://chi.lft.to/lastmile_qr_sc...   
2  1961472602929588766  {'android': 'https://chi.lft.to/lastmile_qr_sc...   
3  1942117929887056608  {'

# Se observan datos duplciados

In [2]:
# 1️⃣ Detectar duplicados en 'name'
dupes = stations[stations['name'].duplicated(keep=False)]

print(f"Duplicados detectados en stations_bikes['name']: {dupes['name'].nunique()} nombres con conflicto")
print(dupes.sort_values('name')[['name', 'station_id', 'short_name']])

# 2️⃣ Resolver duplicados: quedarnos con el que tenga short_name válido
stations_bikes_clean = (
    stations
    .sort_values(by=['name', 'short_name'], ascending=[True, False])  # Los que tienen short_name no NaN primero
    .drop_duplicates(subset='name', keep='first')                     # Conservar el primero (el válido)
)

# 3️⃣ Comprobar resultado
print(f"\nAntes: {len(stations)} filas")
print(f"Después de limpiar duplicados: {len(stations_bikes_clean)} filas")
print(f"Duplicados eliminados: {len(stations) - len(stations_bikes_clean)}")


Duplicados detectados en stations_bikes['name']: 4 nombres con conflicto
                           name           station_id short_name
149      Elston Ave & George St  2111730464161852178   CHI02084
1541     Elston Ave & George St  1594046452527747848        NaN
145      Indiana Ave & 133rd St  1978857650118994914   CHI01933
1869     Indiana Ave & 133rd St  1448642188027369086        NaN
77    Rockwell St & Fletcher St  2108374001471259348   CHI02082
1091  Rockwell St & Fletcher St  1594046422462976736        NaN
50        Western Ave & Lake St  1967727360320698512   CHI01848
1144      Western Ave & Lake St  1594046379513303720        NaN

Antes: 1916 filas
Después de limpiar duplicados: 1912 filas
Duplicados eliminados: 4


# Normalización de las estaciones

In [9]:
from sklearn.preprocessing import LabelEncoder

le_station = LabelEncoder()

stations_bikes_clean["station_idx"] = le_station.fit_transform(
    stations_bikes_clean["station_id"]
)


In [10]:
# Se observan las etiquetas
le_station.classes_

array(['03a1610c-3ba0-4b85-b512-bc26791cb7ee',
       '080ab938-2c1b-4fcf-a83d-74c5ae6ff305',
       '0eeb6baf-c653-4a1a-af17-99f686116b89', ...,
       'ee925757-31ce-41ad-9bb7-73b1bb363616',
       'f0f9f5dc-3d3f-4a32-aa3e-b87566e44de9',
       'ff21c158-de98-46c6-9ef7-7082bdb7c634'],
      shape=(1912,), dtype=object)

In [11]:
# Se observa una de las estaciones
le_station.transform(["a3a44476-a135-11e9-9cda-0a87ae2ba916"])[0]

np.int64(1331)

In [12]:
import pickle

with open('../../data/bicycles/le_station_encoder.pkl', 'wb') as f:
    pickle.dump(le_station, f)

In [13]:
stations_bikes_clean.to_pickle("../../data/bicycles/df_bicycles_stations.pk1")